In [ ]:
# Importar as bibliotecas necessárias
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Carregar o dataset
from google.colab import drive
drive.mount('/content/drive')

# Substitua o caminho abaixo pelo caminho correto para o seu arquivo CSV
file_path = '/content/drive/MyDrive/Dissertacao_Final/base_nirs_refletancia_final.csv'
data = pd.read_csv(file_path)

# Definir o atributo alvo e os atributos de entrada
X = data.iloc[:, 6:]  # Selecionar a partir da 7ª coluna até o final
y = data['TOC']       # Atributo alvo "TOC"

# Dividir o dataset em conjunto de treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalizar os dados
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Seleção de Atributos com Base em Filtro - Correlação e ANOVA
# 1. Usando ANOVA F-valor
anova_selector = SelectKBest(score_func=f_regression, k=10)
X_train_anova = anova_selector.fit_transform(X_train_scaled, y_train)
X_test_anova = anova_selector.transform(X_test_scaled)

# 2. Usando Correlação de Pearson
correlation = X.corrwith(y)
top_10_corr_features = correlation.abs().sort_values(ascending=False).head(10).index
X_train_corr = X_train_scaled[:, X.columns.get_indexer(top_10_corr_features)]
X_test_corr = X_test_scaled[:, X.columns.get_indexer(top_10_corr_features)]

# Treinamento e Avaliação com ANOVA
# Criar e treinar o modelo de regressão linear usando os atributos selecionados por ANOVA
linear_model_anova = LinearRegression()
linear_model_anova.fit(X_train_anova, y_train)

# Calcular o R² médio após validação cruzada
cv_scores_anova = cross_val_score(linear_model_anova, X_train_anova, y_train, cv=5, scoring='r2')
mean_cv_score_anova = cv_scores_anova.mean()
print(f"R² médio após validação cruzada (ANOVA, 10 atributos): {mean_cv_score_anova:.4f}")

# Calcular o erro médio absoluto
y_pred_anova = linear_model_anova.predict(X_test_anova)
mae_anova = mean_absolute_error(y_test, y_pred_anova)
print(f"Erro Médio Absoluto (ANOVA, 10 atributos): {mae_anova:.4f}")

# Criar a equação de estimação reduzida para 10 atributos com ANOVA
coefficients_anova = linear_model_anova.coef_
intercept_anova = linear_model_anova.intercept_

equation_anova = f"TOC = {intercept_anova:.4f}"
for feature, coef in zip(X.columns[anova_selector.get_support()], coefficients_anova):
    equation_anova += f" + ({coef:.4f})*{feature}"

print(f"Equação de Estimação Reduzida (ANOVA, 10 atributos): {equation_anova}")

# Treinamento e Avaliação com Correlação
# Criar e treinar o modelo de regressão linear usando os atributos selecionados por Correlação
linear_model_corr = LinearRegression()
linear_model_corr.fit(X_train_corr, y_train)

# Calcular o R² médio após validação cruzada
cv_scores_corr = cross_val_score(linear_model_corr, X_train_corr, y_train, cv=5, scoring='r2')
mean_cv_score_corr = cv_scores_corr.mean()
print(f"R² médio após validação cruzada (Correlação, 10 atributos): {mean_cv_score_corr:.4f}")

# Calcular o erro médio absoluto
y_pred_corr = linear_model_corr.predict(X_test_corr)
mae_corr = mean_absolute_error(y_test, y_pred_corr)
print(f"Erro Médio Absoluto (Correlação, 10 atributos): {mae_corr:.4f}")

# Criar a equação de estimação reduzida para 10 atributos com Correlação
coefficients_corr = linear_model_corr.coef_
intercept_corr = linear_model_corr.intercept_

equation_corr = f"TOC = {intercept_corr:.4f}"
for feature, coef in zip(top_10_corr_features, coefficients_corr):
    equation_corr += f" + ({coef:.4f})*{feature}"

print(f"Equação de Estimação Reduzida (Correlação, 10 atributos): {equation_corr}")
